# 20261005. Синтаксис в NLP

План занятия:
- синтаксис в первом приближении
- NLP-библиотека `spaCy`
- Universal Dependencies, формат CoNLL-U и библиотека `pyconll`

## Зачем нам инструменты, анализирующие синтаксис предложения?

После того как мы токенизировали текст и получили какой-то морфологический анализ, нам может понадобиться **синтаксическая** информация о предложении:
- Какой в нем порядок слов?
- Какие грамматические отношения между токенами есть в предложении? Кто с кем согласуется? Кто от кого зависит?
- и т.д.

Для того чтобы уметь отвечать на эти вопросы, нам нужен **синтаксический парсер**.

Их есть много и разных. Более детальный обзор вы можете почитать на [Хабре](https://habr.com/ru/companies/sberbank/articles/418701/).

Синтаксис – это один из уровней языка, благодаря которому предложения и тексты становятся грамматичными и связными друг с другом. Знания о синтаксисе помогают и в генерации текста, и в задаче QA, и т.д.

В рамках курса «Теория языка» у вас будет отдельный блок про синтаксис, где вы узнаете про основные сложности и тонкости, а пока будут некоторые упрощения и спойлеры.

## Базовые понятия

Каждое предложение состоит из слов, которые как-то друг с другом связаны. В синтаксисе есть два подхода по тому, как предложение может быть проанализировано:
- Грамматика зависимостей
- Грамматика составляющих

Дать определение такой сущности как составляющая – непросто. На синтаксисе вы будете обсуждать критерии составляющих (спойлер: разные критерии дают разные вердикты).

Синтаксические парсеры в NLP в основном работают с **грамматикой зависимостей**: каждое предложение представляется в виде дерева, где связь между токенами – это грамматическое отношение между ними (или зависимость).

Пример такого грамматического отношения: прямое дополнение – зависимость между кем и кем?

Иллюстрация, сравнивающая два подхода:

![](https://upload.wikimedia.org/wikipedia/commons/0/0d/Wearetryingtounderstandthedifference_%282%29.jpg)

## Библиотека `spaCy`

### Основная информация

Документация – [здесь](https://spacy.io/).

Что умеет эта библиотека?
- морфологический анализ (см. предыдущий семинар)
- строить и визуализировать деревья зависимостей

И многое другое. И для русского тоже.

БОНУС: есть [курс](https://course.spacy.io/en/) по работе с библиотекой от ее создателей.

### Пример работы

Установка библиотеки

In [1]:
!pip install spacy

Для работы с русским языком нужно загрузить модель.

Просто предупреждение: как показывают опыт и практика, с установкой на разных системах и питонах могут быть проблемы.

In [2]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 66.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


После загрузки модели мы можем ее импортировать и передавать ей тексты для разбора.

In [3]:
import spacy
from spacy.lang.ru.examples import sentences

nlp = spacy.load("ru_core_news_sm")

Терминология в грамматике зависимостей:
- Токен, от которого зависит текущий, – его родитель, или вершина (head).
- Токен, который зависит от текущего, – его ребенок, или зависимое (dependent).

Для каждого токена мы можем найти его вершину, детей и грамматические отношения между ними всеми. Иными словами, восстановить грамматику зависимостей данного предложения.

У каждого предложения есть корень, или ROOT, от которого начинает строиться грамматика зависимостей. Обычно корнем является предикат.

Список сокращений, доступных в библиотеке, можно найти в этом [глоссарии](https://github.com/explosion/spaCy/blob/master/spacy/glossary.py).

In [4]:
doc = nlp(sentences[0])

for token in doc:
    print(f"Токен: {token.text}")
    print(f"Часть речи: {token.pos_}")
    print(f"Грамматическое отношение к вершине: {token.dep_}")
    print()

Токен: Apple
Часть речи: PROPN
Грамматическое отношение к вершине: nsubj

Токен: рассматривает
Часть речи: VERB
Грамматическое отношение к вершине: ROOT

Токен: возможность
Часть речи: NOUN
Грамматическое отношение к вершине: obj

Токен: покупки
Часть речи: NOUN
Грамматическое отношение к вершине: nmod

Токен: стартапа
Часть речи: NOUN
Грамматическое отношение к вершине: nmod

Токен: из
Часть речи: ADP
Грамматическое отношение к вершине: case

Токен: Соединённого
Часть речи: ADJ
Грамматическое отношение к вершине: amod

Токен: Королевства
Часть речи: PROPN
Грамматическое отношение к вершине: nmod

Токен: за
Часть речи: ADP
Грамматическое отношение к вершине: case

Токен: $
Часть речи: SYM
Грамматическое отношение к вершине: nmod

Токен: 1
Часть речи: NUM
Грамматическое отношение к вершине: appos

Токен: млрд
Часть речи: NOUN
Грамматическое отношение к вершине: punct



Из этого глоссария можно достать расшифровку при помощи `spacy.explain()`.

In [5]:
spacy.explain("NP")

'noun phrase'

Найдем для первого токена его родителя (`.head`)

In [6]:
for token in doc:
    print(f"Токен: {token.text}")
    print(f"Часть речи: {token.pos_}")
    print(f"Грамматическое отношение к вершине: {token.dep_}")
    print(f"Вершина: {token.head}")
    print(f"Часть речи вершины: {token.head.pos_}")
    break

Токен: Apple
Часть речи: PROPN
Грамматическое отношение к вершине: nsubj
Вершина: рассматривает
Часть речи вершины: VERB


и его детей (`.children`).

In [7]:
for token in doc:
    print(f"Токен: {token.text}")
    print(f"Часть речи: {token.pos_}")
    print(f"Дети: {' '.join([child.text for child in token.children])}")
    break

Токен: Apple
Часть речи: PROPN
Дети: 


Если бы у первого токена дети были, то можно было бы еще получить целое поддерево (`.subtree`), начинающееся с него.

Дерево зависимостей можно визуализировать при помощи `displacy`.

In [8]:
from spacy import displacy

displacy.render(doc, style='dep')

Визуализация может быть кастомизирована (цвет, шрифт и т.д.). Подробнее можете почитать [здесь](https://spacy.io/usage/visualizers). В документации приводится вот такое дерево для примера:

In [9]:
options = {"compact": True,
           "bg": "#09a3d5",
           "color": "white",
           "font": "Source Sans Pro"
           }

displacy.render(doc, style="dep", options=options)

## Практика - 1

Задание №0 (Подготовительное). До этого момента мы работали в `spaCy` с русским языком. Установите модель для английского. В этом вам поможет [документация](https://spacy.io/usage/models).

In [10]:
# Ваш код здесь

Задание №1.1 (Синтаксическая неоднозначность). Иногда предложения могут быть сформулированы неоднозначно. На самом деле, это значит, что мы по-разному можем поделить предложение на *составляющие*.

Возьмите предложения с [синтаксической неоднозначностью](https://en.wikipedia.org/wiki/Syntactic_ambiguity) (можно придумать свои, если хотите):
- *John saw the man on the mountain with a telescope.*
- *I'm glad I'm a man, and so is Lola.*

Вопрос: Какая интерпретация может быть у этих предложений?

Теперь проверьте, какой разбор делает `spacy`. Выведите для каждого токена его вершину и грамматическое отношение, а затем визуализируйте дерево.

In [11]:
# Ваш код здесь

Задание №1.2 (Однородные члены предложения). Предложения могут быть устроены таким образом, что как будто бы несколько слов имеют одну и ту же вершину с одним и тем же грамматическим отношением. Иными словами – однородные члены предложения.

Как такие ситуации обрабатывает библиотека `spaCy` в контексте грамматики зависимостей?

Чтобы ответить на этот вопрос, проанализируйте предложение, где есть однородные члены предложения (придумайте самостоятельно). Выведите для каждого токена вершину и грамматическое отношение, а затем визуализируйте дерево.

In [12]:
# Ваш код здесь

## Universal Dependencies и формат CoNLL-U

### Основная информация

[**Universal Dependencies**](https://universaldependencies.org/) – это проект по унификации разметки синтаксических корпусов (трибанков) в рамках грамматики зависимостей. Сейчас он насчитывает более 200 трибанков на более чем 150 языках.

Когда мы начинаем работать с большим количеством языков, мы сталкиваемся с проблемой того, что, например, разделение на части речи неуниверсально. Где-то есть артикли (например, в английском) и нужно описывать связь между ним и существительным, с которым оно связано, а где-то их нет (например, в русском). Да и вообще – все грамматики написаны по-разному разными людьми.

А если мы хотим создать мультилингвальный универсальный синтаксический парсер, то как раз и нужна унифицированная разметка, которая поможет сопоставлять разные грамматики.

Героическим создателям Universal Dependencies удалось договориться между собой и разметить все корпусы, которые имелись в их распоряжении, в едином формате.

### Как выглядит корпус в UD?

У каждого трибанка – своя страница на сайте самого проекта и свой репозиторий. Там вы можете найти информацию о:
- Источнике данных
- Разметке (морфологические теги, их значения)
- Лицензии (помните нашу презентацию и предостережения), авторах и т.д.
- Файлах с самим корпусом

Вот [пример](https://universaldependencies.org/treebanks/ru_syntagrus/index.html) того, как это выглядит для SynTagRus, одного из крупных трибанков на русском языке.

По сути, каждый трибанк – это текстовый файл в определенном формате, CoNLL-U (расширение у таких файлов соответствующее, `.conllu`). Оно происходит от названия конференции «Conference on Computational Natural Language Learning», в рамках которой проводились и проводятся соревнования по парсингу.

Вот пример одного предложения, размеченного в таком формате.

```
# sent_id = 2
# text = I have no clue.
1   I       I       PRON    PRP   Case=Nom|Number=Sing|Person=1     2   nsubj   _   _
2   have    have    VERB    VBP   Number=Sing|Person=1|Tense=Pres   0   root    _   _
3   no      no      DET     DT    PronType=Neg                      4   det     _   _
4   clue    clue    NOUN    NN    Number=Sing                       2   obj     _   SpaceAfter=No
5   .       .       PUNCT   .     _                                 2   punct   _   _
```

Кратко об этой разметке:
- Предложения в корпусе отделяются двумя пустыми строками
- Каждое слово предложения – на своей новой строчке
- Каждая строчка – это $10$ полей. Номер слова, его лемма, его вершина и т.д. Подробнее про формат – [здесь](https://universaldependencies.org/format.html).
- Все поля отделены друг от друга табами

### Как работать с CoNLL-U?

Для парсинга CoNLL-U существуют библиотеки, и сегодня мы разберемся, как работать с модулем `pyconll`. Мы будем работать с файлом `training_data.conllu` – можете скачать его по [ссылке](https://raw.githubusercontent.com/hse-ling-python/hse-ling_26-27/refs/heads/main/files/training_data.conllu). В нем отобраны несколько случайных предложений из SynTagRus.

Установка библиотеки.

Документация и функции отличаются для версий библиотеки 3.x и 4.x. В этой тетрадке будет использоваться версия 3.3.1. Если у вас версия питона >=3.14, то скорее всего без указания версии будет установлена версия 4.x. Имейте это в виду и открывайте нужную версию документации.

Документация версии 3.3.1 находится [здесь](https://pyconll.readthedocs.io/en/3.3.1/starting.html).

In [13]:
!pip install pyconll==3.3.1

Импорт библиотеки

In [14]:
import pyconll

Загрузка файла в формате `.conllu` при помощи метода `.load_from_file`.

In [15]:
data = pyconll.load_from_file("training_data.conllu")

Библиотека позволяет итерироваться по предложениям и по токенам в этих предложениях при помощи обычных циклов `for`.

In [16]:
for sentence in data:
    for token in sentence:
        pass

Для каждого предложения мы можем:
- Узнать его `id`
- Вывести сам текст (без решеток, комментариев и т.д.)

In [17]:
for sentence in data:
    print(f"ID: {sentence.id}")
    print(sentence.text)
    break

ID: uch-nauch--encicl_phil--encicl94-13
Но особенно похожи на предложения естественного языка формулы математической логики -- науки, в которой изучается строение точных рассуждений, в первую очередь математических, и при этом используются математические же методы.


Для каждого токена в предложении мы можем вывести следующую информацию:
- Номер / ID
- Текст
- Лемма
- Часть речи
- Номер вершины
- Грамматическое отношение (зависимость) по отношению к вершине
- Морфологическая разметка

In [18]:
for sentence in data:
    for token in sentence:
        print(f"ID: {token.id}")
        print(f"Токен: {token.form}")
        print(f"Лемма: {token.lemma}")
        print(f"Часть речи: {token.upos}")
        print(f"ID вершины: {token.head}")
        print(f"Зависимость: {token.deprel}")
        print(f"Морфологическая разметка: {token.feats}")
        print()
    break

ID: 1
Токен: Но
Лемма: но
Часть речи: CCONJ
ID вершины: 3
Зависимость: cc
Морфологическая разметка: {}

ID: 2
Токен: особенно
Лемма: особенно
Часть речи: ADV
ID вершины: 3
Зависимость: advmod
Морфологическая разметка: {'Degree': {'Pos'}}

ID: 3
Токен: похожи
Лемма: похожий
Часть речи: ADJ
ID вершины: 0
Зависимость: root
Морфологическая разметка: {'Degree': {'Pos'}, 'Number': {'Plur'}, 'Variant': {'Short'}}

ID: 4
Токен: на
Лемма: на
Часть речи: ADP
ID вершины: 5
Зависимость: case
Морфологическая разметка: {}

ID: 5
Токен: предложения
Лемма: предложение
Часть речи: NOUN
ID вершины: 3
Зависимость: obl
Морфологическая разметка: {'Animacy': {'Inan'}, 'Case': {'Acc'}, 'Gender': {'Neut'}, 'Number': {'Plur'}}

ID: 6
Токен: естественного
Лемма: естественный
Часть речи: ADJ
ID вершины: 7
Зависимость: amod
Морфологическая разметка: {'Case': {'Gen'}, 'Degree': {'Pos'}, 'Gender': {'Masc'}, 'Number': {'Sing'}}

ID: 7
Токен: языка
Лемма: язык
Часть речи: NOUN
ID вершины: 5
Зависимость: nmod
Морфолог

Если нас интересует конкретный токен, и мы знаем его ID, то мы можем получить к нему доступ при помощи обычной списковой индексации.

In [19]:
for sentence in data:
    print(sentence[2].form)
    break

похожи


## Практика - 2

Задание №0 (Подготовительное). Скачайте по [ссылке](https://raw.githubusercontent.com/hse-ling-python/hse-ling_26-27/refs/heads/main/files/practice_data.conllu) файл `practice_data.conllu` и прочитайте его при помощи модуля `pyconll`.

In [20]:
# Ваш код здесь

Задание №2.1. Сколько в этом файле предложений? Какая средняя длина предложений в токенах? Какое предложение самое длинное? Какое предложение самое короткое?

Ответьте на эти вопросы, используя модуль `pyconll` и циклы `for`.

In [21]:
# Ваш код здесь

Задание №2.2. Сколько частей речи представлено в этом корпусе? Сделайте частотный словарь.

In [22]:
# Ваш код здесь

Задание №2.3. Сколько грамматических отношений представлено в этом корпусе? Сделайте частотный словарь.

In [23]:
# Ваш код здесь

Задание №2.4. Выведите все именные группы (NP). Для упрощения будем считать, что это любое существительное и зависимые от него токены (дети).

In [24]:
# Ваш код здесь

Задание №2.5. Основываясь на результате предыдущего задания – какие части речи в основном становятся зависимыми от существительных в данном корпусе?

In [25]:
# Ваш код здесь

Задание №2.6. В каждом предложении найдите подлежащее и прямое дополнение (при их наличии) и выведите их. Какой порядок слов в этих предложениях?

In [26]:
# Ваш код здесь